In [ ]:
from pathlib import Path
import pandas as pd
from ydata_profiling import ProfileReport

DATA_DIR = Path("data/raw")
OUT_DIR = Path("reports/automated_profiling")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Profile all small files
files = {
    "courses":             "courses.csv",
    "assessments":         "assessments.csv",
    "studentInfo":         "studentInfo.csv",
    "studentRegistration": "studentRegistration.csv",
    "studentAssessment":   "studentAssessment.csv",
    "vle":                 "vle.csv",
}

for name, filename in files.items():
    df = pd.read_csv(DATA_DIR / filename)
    profile = ProfileReport(df, title=f"OULAD automated profile - {name}", explorative=True)
    profile.to_file(OUT_DIR / f"{name}_profile.html")
    print(f"Done: {name}")

# studentVle is too large (10M rows) — sample 100k rows for the raw scan
student_vle = pd.read_csv(DATA_DIR / "studentVle.csv")
sample_vle = student_vle.sample(n=100000, random_state=42)

ProfileReport(
    sample_vle,
    title="OULAD automated profile - studentVle sample",
    minimal=True
).to_file(OUT_DIR / "studentVle_sample_profile.html")
print("Done: studentVle sample")

# Profile the aggregated enrolment-level VLE features
vle_features = (
    student_vle
    .groupby(["code_module", "code_presentation", "id_student"])
    .agg(
        total_clicks=("sum_click", "sum"),
        active_days=("date", "nunique"),
        avg_clicks_per_day=("sum_click", "mean")
    )
    .reset_index()
)

ProfileReport(
    vle_features,
    title="OULAD automated profile - VLE enrolment features",
    explorative=True
).to_file(OUT_DIR / "vle_features_profile.html")
print("Done: VLE enrolment features")
print(f"\nAll reports saved to: {OUT_DIR}")

ModuleNotFoundError: No module named 'pkg_resources'